In [2]:
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

In [3]:
df = pd.read_excel('lalo.xlsx')

In [4]:
result_df = pd.DataFrame(columns = ['Address', 'Latitude', 'Longitude'])

In [5]:
df['경도'] = df['경도'].astype(str)
df['위도'] = df['위도'].astype(str)

In [6]:
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

apiurl = "https://api.vworld.kr/req/address?"

result_df = pd.DataFrame(columns=['Address', 'Latitude', 'Longitude'])

for i in range(df.shape[0]):
    lat = df.loc[i, '경도']
    lon = df.loc[i, '위도']
    point = lat + ',' + lon

    params = {
        "service": "address",
        "request": "getaddress",
        "crs": "epsg:4326",
        "point": point,
        "format": "json",
        "type": "road",
        "key": "6278C156-1D31-3D8C-9B1F-D5A8645F8B43"
    }

    session = requests.Session()
    retry = Retry(connect=3, backoff_factor=0.5)
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)

    try:
        response = session.get(apiurl, params=params, timeout=10)
        response.raise_for_status()
        jsondata = response.json()

        print(jsondata)  # 디버깅용 응답 출력

        response_data = jsondata.get('response', {})
        result_data = response_data.get('result', [])

        if result_data:
            address = result_data[0].get('text', '주소 없음')
        else:
            address = '주소 없음'

        result_df = pd.concat([
            result_df,
            pd.DataFrame({'Address': [address],
                          'Latitude': [lat],
                          'Longitude': [lon]})
        ], ignore_index=True)

    except requests.exceptions.RequestException as e:
        print("요청 실패:", e)
    except Exception as e:
        print("예외 발생:", e)


{'response': {'service': {'name': 'address', 'version': '2.0', 'operation': 'getaddress', 'time': '7(ms)'}, 'status': 'NOT_FOUND', 'input': {'point': {'x': '128.5265', 'y': '35.197885'}, 'crs': 'epsg:4326', 'type': 'road'}}}
{'response': {'service': {'name': 'address', 'version': '2.0', 'operation': 'getaddress', 'time': '8(ms)'}, 'status': 'NOT_FOUND', 'input': {'point': {'x': '128.500675', 'y': '35.198701'}, 'crs': 'epsg:4326', 'type': 'road'}}}
{'response': {'service': {'name': 'address', 'version': '2.0', 'operation': 'getaddress', 'time': '7(ms)'}, 'status': 'NOT_FOUND', 'input': {'point': {'x': '128.490488', 'y': '35.231401'}, 'crs': 'epsg:4326', 'type': 'road'}}}
{'response': {'service': {'name': 'address', 'version': '2.0', 'operation': 'getaddress', 'time': '6(ms)'}, 'status': 'NOT_FOUND', 'input': {'point': {'x': '128.5182', 'y': '35.2284'}, 'crs': 'epsg:4326', 'type': 'road'}}}
{'response': {'service': {'name': 'address', 'version': '2.0', 'operation': 'getaddress', 'time': 

In [7]:
result_df

,Address,Latitude,Longitude
0,주소 없음,128.5265,35.197885
1,주소 없음,128.500675,35.198701
2,주소 없음,128.490488,35.231401
3,주소 없음,128.5182,35.2284
4,주소 없음,128.4949,35.2478
...,...,...,...
107,주소 없음,128.6941,35.2741
108,주소 없음,128.6749,35.2668
109,주소 없음,128.663063,35.274367
110,주소 없음,128.6548,35.2761
